In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

from openpyxl import load_workbook
from tqdm.auto import tqdm

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SCAN_FILE = Path(
    "data/idx_financial_file_scan.csv"
)

OUTPUT_FILE = Path(
    "data/idx_financial_metric_candidates.csv"
)

print("Scan file exists:", SCAN_FILE.exists())

Scan file exists: True


In [3]:
scan_df = pd.read_csv(
    SCAN_FILE
)

print("Total scanned XLSX:", len(scan_df))

display(
    scan_df.head()
)

Total scanned XLSX: 16003


,ticker,year,quarter,file_name,file_path,size_mb,is_valid_size,has_zip_signature,can_open_excel,sheet_count,sheet_names,scan_status,error_type,error_message
0,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.338,True,True,True,32.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
1,ZONE,2025,Q1,ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.340,True,True,True,33.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
2,ZINC,2025,Q1,ZINC_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.339,True,True,True,30.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
3,ZATA,2025,Q1,ZATA_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.304,True,True,True,26.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
4,YUPI,2025,Q1,YUPI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.342,True,True,True,33.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN


In [4]:
valid_files_df = (
    scan_df[
        scan_df["scan_status"] == "VALID"
    ]
    .copy()
)

print(
    "Valid XLSX files:",
    len(valid_files_df)
)

print(
    "Unique tickers:",
    valid_files_df["ticker"].nunique()
)

display(
    valid_files_df.head()
)

Valid XLSX files: 15821
Unique tickers: 948


,ticker,year,quarter,file_name,file_path,size_mb,is_valid_size,has_zip_signature,can_open_excel,sheet_count,sheet_names,scan_status,error_type,error_message
0,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.338,True,True,True,32.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
1,ZONE,2025,Q1,ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.340,True,True,True,33.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
2,ZINC,2025,Q1,ZINC_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.339,True,True,True,30.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
3,ZATA,2025,Q1,ZATA_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.304,True,True,True,26.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
4,YUPI,2025,Q1,YUPI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.342,True,True,True,33.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN


In [5]:
def resolve_file_path(path_text):
    path = Path(path_text)

    if path.exists():
        return path

    alternative = Path.cwd() / path

    if alternative.exists():
        return alternative

    return None

In [6]:
def normalize_text(value):

    if value is None:
        return ""

    text = str(value)

    text = text.strip().lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text

In [ ]:
METRIC_KEYWORDS = {

    "revenue": [
        "revenue",
        "revenues",
        "sales and revenue",
        "net sales",
        "sales",
        "pendapatan",
        "penjualan",
        "pendapatan usaha"
    ],

    "gross_profit": [
        "gross profit",
        "gross income",
        "laba bruto",
        "laba kotor"
    ],

    "operating_profit": [
        "operating profit",
        "operating income",
        "profit from operations",
        "income from operations",
        "laba usaha",
        "laba operasi"
    ],

    "net_income": [
        "net income",
        "net profit",
        "profit for the year",
        "profit for the period",
        "profit for the current period",
        "laba bersih",
        "laba tahun berjalan",
        "laba periode berjalan"
    ],

    "total_assets": [
        "total assets",
        "jumlah aset"
    ],

    "total_liabilities": [
        "total liabilities",
        "jumlah liabilitas"
    ],

    "cash": [
        "cash and cash equivalents",
        "cash and cash equivalent",
        "cash and bank",
        "kas dan setara kas"
    ],

    "operating_cash_flow": [
        "net cash flows from operating activities",
        "net cash provided by operating activities",
        "cash flows from operating activities",
        "arus kas bersih dari aktivitas operasi",
        "kas neto diperoleh dari aktivitas operasi",
        "kas neto digunakan untuk aktivitas operasi"
    ]
}

In [8]:
SKIP_SHEETS = {
    "context",
    "inlinexbrl",
    "hidden",
    "token"
}


def should_skip_sheet(sheet_name):

    name = str(
        sheet_name
    ).strip().lower()

    if name in SKIP_SHEETS:
        return True

    return False

In [9]:
def is_numeric_value(value):

    if value is None:
        return False

    if isinstance(
        value,
        (int, float, np.integer, np.floating)
    ):

        if pd.isna(value):
            return False

        return True

    return False

In [10]:
def extract_numeric_candidates(row_values):

    numeric_candidates = []

    for column_index, value in enumerate(
        row_values
    ):

        if is_numeric_value(value):

            numeric_candidates.append({
                "column_index": column_index,
                "value": value
            })

    return numeric_candidates

In [11]:
def match_metric(text):

    normalized = normalize_text(
        text
    )

    if not normalized:
        return None, None

    for metric, keywords in (
        METRIC_KEYWORDS.items()
    ):

        for keyword in keywords:

            keyword_norm = normalize_text(
                keyword
            )

            if normalized == keyword_norm:
                return metric, keyword

    return None, None

In [12]:
def extract_metric_candidates_from_file(row):

    file_path = resolve_file_path(
        row["file_path"]
    )

    results = []

    if file_path is None:
        return results

    try:

        workbook = load_workbook(
            file_path,
            read_only=True,
            data_only=True
        )

        for sheet_name in workbook.sheetnames:

            if should_skip_sheet(
                sheet_name
            ):
                continue

            worksheet = workbook[
                sheet_name
            ]

            for row_number, cells in enumerate(
                worksheet.iter_rows(
                    values_only=True
                ),
                start=1
            ):

                if not cells:
                    continue

                for column_index, cell_value in enumerate(
                    cells
                ):

                    if not isinstance(
                        cell_value,
                        str
                    ):
                        continue

                    metric, matched_keyword = (
                        match_metric(
                            cell_value
                        )
                    )

                    if metric is None:
                        continue

                    numeric_candidates = (
                        extract_numeric_candidates(
                            cells
                        )
                    )

                    results.append({

                        "ticker":
                            row["ticker"],

                        "year":
                            row["year"],

                        "quarter":
                            row["quarter"],

                        "metric":
                            metric,

                        "matched_keyword":
                            matched_keyword,

                        "source_label":
                            cell_value,

                        "source_sheet":
                            sheet_name,

                        "row_number":
                            row_number,

                        "label_column":
                            column_index,

                        "numeric_candidate_count":
                            len(
                                numeric_candidates
                            ),

                        "numeric_candidates":
                            json.dumps(
                                numeric_candidates,
                                ensure_ascii=False,
                                default=str
                            ),

                        "source_file":
                            row["file_name"],

                        "source_path":
                            row["file_path"]
                    })

        workbook.close()

    except Exception as e:

        results.append({

            "ticker":
                row["ticker"],

            "year":
                row["year"],

            "quarter":
                row["quarter"],

            "metric":
                None,

            "matched_keyword":
                None,

            "source_label":
                None,

            "source_sheet":
                None,

            "row_number":
                None,

            "label_column":
                None,

            "numeric_candidate_count":
                None,

            "numeric_candidates":
                None,

            "source_file":
                row["file_name"],

            "source_path":
                row["file_path"],

            "extraction_error":
                f"{type(e).__name__}: {e}"
        })

    return results

In [13]:
test_df = (
    valid_files_df[
        valid_files_df["ticker"].isin(
            [
                "AADI",
                "AMRT",
                "BBRI",
                "KLBF",
                "TLKM"
            ]
        )
    ]
    .sort_values(
        ["ticker", "year", "quarter"],
        ascending=[
            True,
            False,
            False
        ]
    )
    .groupby(
        "ticker",
        as_index=False
    )
    .first()
)

display(
    test_df[
        [
            "ticker",
            "year",
            "quarter",
            "file_name"
        ]
    ]
)

,ticker,year,quarter,file_name
0,AADI,2025,Q1,AADI_2025_Q1_FS.xlsx
1,AMRT,2024,Q4,AMRT_2024_Q4_FS.xlsx
2,BBRI,2025,Q1,BBRI_2025_Q1_FS.xlsx
3,KLBF,2025,Q1,KLBF_2025_Q1_FS.xlsx
4,TLKM,2025,Q1,TLKM_2025_Q1_FS.xlsx


In [14]:
test_results = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Testing metric extraction",
    unit="file"
):

    file_results = (
        extract_metric_candidates_from_file(
            row
        )
    )

    test_results.extend(
        file_results
    )


test_candidates_df = pd.DataFrame(
    test_results
)

print(
    "Candidate matches:",
    len(test_candidates_df)
)

display(
    test_candidates_df
)

Testing metric extraction:   0%|          | 0/5 [00:00<?, ?file/s]

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Testing metric extraction: 100%|██████████| 5/5 [00:04<00:00,  1.08file/s]

Candidate matches: 59


,ticker,year,quarter,metric,matched_keyword,source_label,source_sheet,row_number,label_column,numeric_candidate_count,numeric_candidates,source_file,source_path
0,AADI,2025,Q1,cash,kas dan setara kas,Kas dan setara kas,1210000,8,0,2,"[{""column_index"": 1, ""value"": 1358333}, {""colu...",AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
1,AADI,2025,Q1,cash,cash and cash equivalents,Cash and cash equivalents,1210000,8,3,2,"[{""column_index"": 1, ""value"": 1358333}, {""colu...",AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
2,AADI,2025,Q1,total_assets,jumlah aset,Jumlah aset,1210000,128,0,2,"[{""column_index"": 1, ""value"": 5827769}, {""colu...",AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
3,AADI,2025,Q1,total_assets,total assets,Total assets,1210000,128,3,2,"[{""column_index"": 1, ""value"": 5827769}, {""colu...",AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
4,AADI,2025,Q1,total_liabilities,jumlah liabilitas,Jumlah liabilitas,1210000,247,0,2,"[{""column_index"": 1, ""value"": 2339594}, {""colu...",AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
5,AADI,2025,Q1,total_liabilities,total liabilities,Total liabilities,1210000,247,3,2,"[{""column_index"": 1, ""value"": 2339594}, {""colu...",AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
6,AADI,2025,Q1,total_liabilities,total liabilities and equity,Total liabilities and equity,1210000,273,3,2,"[{""column_index"": 1, ""value"": 5827769}, {""colu...",AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
7,AADI,2025,Q1,revenue,sales and revenue,Sales and revenue,1321000,6,3,2,"[{""column_index"": 1, ""value"": 1164437}, {""colu...",AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
8,AADI,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
9,AADI,2025,Q1,cash,kas dan setara kas,Kas dan setara kas,1610000,8,0,0,[],AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...


In [15]:
if not test_candidates_df.empty:

    test_summary = (
        test_candidates_df[
            test_candidates_df[
                "metric"
            ].notna()
        ]
        .groupby(
            [
                "ticker",
                "metric"
            ]
        )
        .size()
        .reset_index(
            name="matches"
        )
    )

    display(
        test_summary
    )

,ticker,metric,matches
0,AADI,cash,4
1,AADI,operating_cash_flow,1
2,AADI,revenue,3
3,AADI,total_assets,2
4,AADI,total_liabilities,3
5,AMRT,cash,4
6,AMRT,operating_cash_flow,1
7,AMRT,revenue,3
8,AMRT,total_assets,2
9,AMRT,total_liabilities,3


In [16]:
if not test_candidates_df.empty:

    label_summary = (
        test_candidates_df[
            test_candidates_df[
                "metric"
            ].notna()
        ][
            [
                "ticker",
                "metric",
                "source_label",
                "source_sheet",
                "numeric_candidate_count",
                "numeric_candidates"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "ticker",
                "metric"
            ]
        )
    )

    pd.set_option(
        "display.max_colwidth",
        300
    )

    pd.set_option(
        "display.max_rows",
        300
    )

    display(
        label_summary
    )

,ticker,metric,source_label,source_sheet,numeric_candidate_count,numeric_candidates
0,AADI,cash,Kas dan setara kas,1210000,2,"[{""column_index"": 1, ""value"": 1358333}, {""column_index"": 2, ""value"": 1518688}]"
1,AADI,cash,Cash and cash equivalents,1210000,2,"[{""column_index"": 1, ""value"": 1358333}, {""column_index"": 2, ""value"": 1518688}]"
9,AADI,cash,Kas dan setara kas,1610000,0,[]
10,AADI,cash,Cash and cash equivalents,1610000,0,[]
8,AADI,operating_cash_flow,Cash flows from operating activities,1510000,0,[]
7,AADI,revenue,Sales and revenue,1321000,2,"[{""column_index"": 1, ""value"": 1164437}, {""column_index"": 2, ""value"": 1314579}]"
11,AADI,revenue,Sales and revenue,1617000,0,[]
2,AADI,total_assets,Jumlah aset,1210000,2,"[{""column_index"": 1, ""value"": 5827769}, {""column_index"": 2, ""value"": 5992658}]"
3,AADI,total_assets,Total assets,1210000,2,"[{""column_index"": 1, ""value"": 5827769}, {""column_index"": 2, ""value"": 5992658}]"
4,AADI,total_liabilities,Jumlah liabilitas,1210000,2,"[{""column_index"": 1, ""value"": 2339594}, {""column_index"": 2, ""value"": 2629176}]"


In [17]:
full_results = []

total_files = len(valid_files_df)

for _, row in tqdm(
    valid_files_df.iterrows(),
    total=total_files,
    desc="Extracting metric candidates",
    unit="file"
):

    file_results = (
        extract_metric_candidates_from_file(
            row
        )
    )

    full_results.extend(
        file_results
    )

Extracting metric candidates:   0%|          | 0/15821 [00:00<?, ?file/s]e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Extracting metric candidates: 100%|██████████| 15821/15821 [7:13:00<00:00,  1.64s/file]  


In [18]:
metric_candidates_df = pd.DataFrame(
    full_results
)

print(
    "Total candidate rows:",
    len(metric_candidates_df)
)

display(
    metric_candidates_df.head(50)
)

Total candidate rows: 188058


,ticker,year,quarter,metric,matched_keyword,source_label,source_sheet,row_number,label_column,numeric_candidate_count,numeric_candidates,source_file,source_path
0,ZYRX,2025,Q1,cash,kas dan setara kas,Kas dan setara kas,1210000,8,0,2,"[{""column_index"": 1, ""value"": 1888962892}, {""column_index"": 2, ""value"": 6272313805}]",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
1,ZYRX,2025,Q1,cash,cash and cash equivalents,Cash and cash equivalents,1210000,8,3,2,"[{""column_index"": 1, ""value"": 1888962892}, {""column_index"": 2, ""value"": 6272313805}]",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
2,ZYRX,2025,Q1,total_assets,jumlah aset,Jumlah aset,1210000,128,0,2,"[{""column_index"": 1, ""value"": 396429832878}, {""column_index"": 2, ""value"": 392444598785}]",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
3,ZYRX,2025,Q1,total_assets,total assets,Total assets,1210000,128,3,2,"[{""column_index"": 1, ""value"": 396429832878}, {""column_index"": 2, ""value"": 392444598785}]",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
4,ZYRX,2025,Q1,total_liabilities,jumlah liabilitas,Jumlah liabilitas,1210000,247,0,2,"[{""column_index"": 1, ""value"": 98116784695}, {""column_index"": 2, ""value"": 92393710835}]",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
5,ZYRX,2025,Q1,total_liabilities,total liabilities,Total liabilities,1210000,247,3,2,"[{""column_index"": 1, ""value"": 98116784695}, {""column_index"": 2, ""value"": 92393710835}]",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
6,ZYRX,2025,Q1,total_liabilities,total liabilities and equity,Total liabilities and equity,1210000,273,3,2,"[{""column_index"": 1, ""value"": 396429832878}, {""column_index"": 2, ""value"": 392444598785}]",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
7,ZYRX,2025,Q1,revenue,sales and revenue,Sales and revenue,1321000,6,3,2,"[{""column_index"": 1, ""value"": 43381104132}, {""column_index"": 2, ""value"": 34333878046}]",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
8,ZYRX,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
9,ZYRX,2025,Q1,cash,kas dan setara kas,Kas dan setara kas,1610000,8,0,0,[],ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx


In [19]:
# Hapus false mapping:
# "Total liabilities and equity" tidak boleh dianggap total_liabilities

metric_candidates_df = metric_candidates_df[
    ~(
        metric_candidates_df["metric"].eq("total_liabilities")
        &
        metric_candidates_df["source_label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("total liabilities and equity")
    )
].copy()

print("Rows after cleanup:", len(metric_candidates_df))

Rows after cleanup: 173558


In [20]:
display(
    metric_candidates_df[
        metric_candidates_df["source_label"]
        .astype(str)
        .str.lower()
        .eq("total liabilities and equity")
    ]
)

,ticker,year,quarter,metric,matched_keyword,source_label,source_sheet,row_number,label_column,numeric_candidate_count,numeric_candidates,source_file,source_path


In [21]:
from pathlib import Path

OUTPUT_FILE = Path(
    "data/idx_financial_metric_candidates.csv"
)

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

metric_candidates_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved to:", OUTPUT_FILE)
print("Rows saved:", len(metric_candidates_df))

Saved to: data\idx_financial_metric_candidates.csv
Rows saved: 173558
